# Supplementary Figure 10 - Human lymph node StructureMap

            This notebook is a cleaned, English-commented manuscript code file.

            **Provenance.** `CellGPS manuscript/figures/LN dot plot R script` and lymph node t-by-c output folder.

            The notebook keeps the original manuscript data paths where they were found. If a path points to
            `/data/taobo.hu` or `/mnt/taobo.hu`, run the notebook on the A100 server or adapt the path to the
            matching mounted Y-drive location.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
sns.set_theme(context="talk", style="white")


In [ ]:
from sfplot import compute_cophenetic_distances_from_df, plot_cophenetic_heatmap


def compute_coste_structuremap(points: pd.DataFrame, sample_name: str, output_dir: Path):
    """Compute the COSTE/Cell-GPS StructureMap and save the heatmap/table.

    The required table columns are `x`, `y`, and `celltype`. The manuscript uses the
    row cophenetic matrix as the Searcher's D / SSS heatmap.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    row_coph, col_coph = compute_cophenetic_distances_from_df(
        points[["x", "y", "celltype"]],
        x_col="x",
        y_col="y",
        celltype_col="celltype",
        method="average",
        show_corr=False,
    )
    row_coph.to_csv(output_dir / f"StructureMap_table_{sample_name}.csv")
    plot_cophenetic_heatmap(
        row_coph,
        matrix_name="row_coph",
        output_filename=str(output_dir / f"StructureMap_of_{sample_name}.pdf"),
        sample=sample_name,
    )
    return row_coph, col_coph


In [ ]:
XENIUM_DIR = Path("Y:/long/10X_datasets/Xenium/Xenium_5K/Xenium_Prime_Human_Lymph_Node_Reactive_FFPE_outs")
OUTPUT_DIR = Path("outputs/supp_fig_10_lymph_node_structuremap")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cells = pd.read_csv(XENIUM_DIR / "cells.csv.gz")
cell_types = pd.read_csv(XENIUM_DIR / "Xenium_Prime_Human_Lymph_Node_Reactive_FFPE_cell_types.csv")
if len(cell_types.columns) == 2:
    cell_types.columns = ["cell_id", "celltype"]
merged = cells.merge(cell_types[["cell_id", "celltype"]], on="cell_id", how="inner")
points = merged.rename(columns={"x_centroid": "x", "y_centroid": "y"})[["x", "y", "celltype"]].dropna()
row_coph, _ = compute_coste_structuremap(points, "Human_Lymph_Node_Reactive_FFPE", OUTPUT_DIR)

fig, ax = plt.subplots(figsize=(7, 7))
for label, sub in points.groupby("celltype"):
    ax.scatter(sub["x"], sub["y"], s=0.3, label=label, rasterized=True, linewidths=0)
ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
ax.legend(frameon=False, markerscale=6, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.savefig(OUTPUT_DIR / "lymph_node_spatial_map.pdf", bbox_inches="tight")
plt.close(fig)
